In [1]:
import torch

In [2]:
import torch

state_dict = torch.load("/storage_nvme_1/mpioro/pc/model/conslidated_model_state_dict_2.pt", map_location="cpu")


In [5]:
target_model_dict = {k: v for k, v in state_dict.items() if k.startswith("target_model.")}
target_model_dict.keys()

dict_keys(['target_model.embedding', 'target_model.encoder.blocks.0.attention_layer.norm.weight', 'target_model.encoder.blocks.0.attention_layer.layer.q_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.k_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.v_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.o_proj.weight', 'target_model.encoder.blocks.0.ff_layer.norm.weight', 'target_model.encoder.blocks.0.ff_layer.layer.ff_pre_act.weight', 'target_model.encoder.blocks.0.ff_layer.layer.ff_post_act.weight', 'target_model.encoder.blocks.0.ff_layer.layer.gate.weight', 'target_model.encoder.blocks.1.attention_layer.norm.weight', 'target_model.encoder.blocks.1.attention_layer.layer.q_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.k_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.v_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.o_proj.weight', 'target_model.encoder.blocks.1.ff_layer.no

In [29]:
# target_model_dict.keys()
# state_dict.keys()
embedding_dict = {k: v for k, v in state_dict.items() if 'embedding' in k}
print(embedding_dict.keys())

print(embedding_dict['source_model.embedding.weight'].shape)
print(embedding_dict['projections.embedding'].shape)
print(embedding_dict['projections.auxiliary_embedding_weights.weight'].shape)

projected_embedding_weight = embedding_dict['source_model.embedding.weight'] @ embedding_dict['projections.embedding'].T + embedding_dict['projections.auxiliary_embedding_weights.weight']
projected_embedding_weight.shape

dict_keys(['source_model.embedding.weight', 'projections.embedding', 'projections.auxiliary_embedding_weights.weight'])
torch.Size([128256, 4096])
torch.Size([3072, 4096])
torch.Size([128256, 3072])


torch.Size([128256, 3072])

In [36]:

# llama.state_dict()
# len(llama)
llama

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [49]:
llama.state_dict().keys()

odict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.2.self_attn.q_proj.weight', 'model.layers.2.self_attn.k_proj.weight', 'model.layers.2.self_attn.v_proj.weight', 'model.layers.2.self_attn.o_proj.weight', 'model.layers.2.mlp.gate_proj.weight', 'mod

In [ ]:
from transformers import AutoConfig
from transformers.models.llama.modeling_llama import LlamaForCausalLM

# 1. change llama config to match projected compression config
# 2. create new llama model with that config
# 3. load projected compression state dict into that model
# 4. save model to some location

# 1. 
conf = AutoConfig.from_pretrained("meta-llama/Llama-3.1-8B")
llama = LlamaForCausalLM(conf)
# conf.num_hidden_layers = # stays the same
# conf.num_key_value_heads = # stays the same
# conf.num_attention_heads = # stays the same
conf.hidden_size = state_dict['target_model.encoder.blocks.0.attention_layer.norm.weight'].shape[0]
conf.intermediate_size = state_dict['target_model.encoder.blocks.0.ff_layer.layer.gate.weight'].shape[0]
# 2.
llama = LlamaForCausalLM(conf)
# 3. now we need to juggle because the state dict keys are different
new_state_dict = {
    'model.embed_tokens.weight': target_model_dict['target_model.embedding.weight'],
    'model.norm.weight': target_model_dict['target_model.head.norm.weight'],
    'lm_head.weight': target_model_dict['target_model.head.linear.weight'],
}

# embed_tokens
for layer in range(conf.num_hidden_layers):
    prefix_src = f'target_model.encoder.blocks.{layer}.'
    prefix_tgt = f'model.layers.{layer}.'

    # attention
    new_state_dict[f'{prefix_tgt}input_layernorm.weight'] = target_model_dict[f'{prefix_src}attention_layer.norm.weight']
    new_state_dict[f'{prefix_tgt}self_attn.q_proj.weight'] = target_model_dict[f'{prefix_src}attention_layer.layer.q_proj.weight']
    new_state_dict[f'{prefix_tgt}self_attn.k_proj.weight'] = target_model_dict[f'{prefix_src}attention_layer.layer.k_proj.weight']
    new_state_dict[f'{prefix_tgt}self_attn.v_proj.weight'] = target_model_dict[f'{prefix_src}attention_layer.layer.v_proj.weight']
    new_state_dict[f'{prefix_tgt}self_attn.o_proj.weight'] = target_model_dict[f'{prefix_src}attention_layer.layer.o_proj.weight']

    # mlp
    new_state_dict[f'{prefix_tgt}mlp.gate_proj.weight'] = target_model_dict[f'{prefix_src}ff_layer.layer.gate.weight']
    new_state_dict[f'{prefix_tgt}mlp.up_proj.weight'] = target_model_dict[f'{prefix_src}ff_layer.layer.ff_pre_act.weight']
    new_state_dict[f'{prefix_tgt}mlp.down_proj.weight'] = target_model_dict[f'{prefix_src}ff_layer.layer.ff_post_act.weight']
    new_state_dict[f'{prefix_tgt}post_attention_layernorm.weight'] = target_model_dict[f'{prefix_src}ff_layer.norm.weight']



# def get_config(state_dict):

#     conf = AutoConfig.from_pretrained("meta-llama/Llama-3.1-8B")
#     # _vocab_size, dmodel = embedding_weight.shape
#     conf.num_hidden_layers = cfg.common.n_blocks
#     conf.num_key_value_heads = cfg.common.kv_heads
#     conf.num_attention_heads = cfg.common.q_heads
#     # conf._name_or_path = f"{PROJECT_ORG}/{MODEL_NAME}"
#     conf.hidden_size = cfg.common.dmodel
#     conf.intermediate_size =  cfg.common.dff 
#     conf.head_dim = cfg.common.dhead

#     model2 = LlamaForCausalLM(conf)

In [53]:
llama.load_state_dict(new_state_dict, strict=True)

<All keys matched successfully>

In [55]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
device = 'cuda'
llama.to(device)
res = llama.generate(**tokenizer("Hello, my dog is cute", return_tensors="pt").to(device))
print(tokenizer.decode(res[0]))

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|>Hello, my dog is cute and sweet, but he is also a bit shy. He is a bit afraid of strangers, and


In [63]:
hf_save_dir = "/storage_nvme_1/mpioro/pc/model/converted_to_hf"
llama.save_pretrained(hf_save_dir, state_dict=new_state_dict)
tokenizer.save_pretrained(hf_save_dir)

('/storage_nvme_1/mpioro/pc/model/converted_to_hf/tokenizer_config.json',
 '/storage_nvme_1/mpioro/pc/model/converted_to_hf/special_tokens_map.json',
 '/storage_nvme_1/mpioro/pc/model/converted_to_hf/tokenizer.json')

In [75]:
# now evaluate with lm-eval
import json
from lm_eval import evaluator

results = evaluator.simple_evaluate(
    model="hf",
    model_args=f"pretrained={hf_save_dir},tokenizer={hf_save_dir}",
    # model_args=eval_model_args,
    tasks='arc_easy',
    # limit=self.limit,
    device='cuda',
    log_samples=False,
    batch_size=64,
)

with open("/storage_nvme_1/mpioro/pc/model/converted_to_hf/eval_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9501/9501 [00:50<00:00, 187.57it/s]
fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [76]:
results

{'results': {'arc_easy': {'alias': 'arc_easy',
   'acc,none': 0.6666666666666666,
   'acc_stderr,none': 0.009673016668133489,
   'acc_norm,none': 0.6325757575757576,
   'acc_norm_stderr,none': 0.009892552616211405}},
 'group_subtasks': {'arc_easy': []},
 'configs': {'arc_easy': {'task': 'arc_easy',
   'tag': ['ai2_arc'],
   'dataset_path': 'allenai/ai2_arc',
   'dataset_name': 'ARC-Easy',
   'training_split': 'train',
   'validation_split': 'validation',
   'test_split': 'test',
   'doc_to_text': 'Question: {{question}}\nAnswer:',
   'doc_to_target': '{{choices.label.index(answerKey)}}',
   'unsafe_code': False,
   'doc_to_choice': '{{choices.text}}',
   'description': '',
   'target_delimiter': ' ',
   'fewshot_delimiter': '\n\n',
   'num_fewshot': 0,
   'metric_list': [{'metric': 'acc',
     'aggregation': 'mean',
     'higher_is_better': True},
    {'metric': 'acc_norm', 'aggregation': 'mean', 'higher_is_better': True}],
   'output_type': 'multiple_choice',
   'repeats': 1,
   'shou

In [72]:
tm = evaluator.TaskManager()
tm.all_tasks

with open('/home/mpioro/llmrandom_cemetery/Llame_PC_2026-01-21_19-46-08/eval_results.json', 'w') as f:
    f.write(str(tm.all_tasks))

In [50]:
# print(state_dict.keys())
print(state_dict['target_model.encoder.blocks.0.ff_layer.layer.gate.weight'].shape)
# state_dict['target_model.encoder.blocks.0.mlp.fc1.weight']

target_model_dict.keys()

torch.Size([9216, 3072])


dict_keys(['target_model.encoder.blocks.0.attention_layer.norm.weight', 'target_model.encoder.blocks.0.attention_layer.layer.q_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.k_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.v_proj.weight', 'target_model.encoder.blocks.0.attention_layer.layer.o_proj.weight', 'target_model.encoder.blocks.0.ff_layer.norm.weight', 'target_model.encoder.blocks.0.ff_layer.layer.ff_pre_act.weight', 'target_model.encoder.blocks.0.ff_layer.layer.ff_post_act.weight', 'target_model.encoder.blocks.0.ff_layer.layer.gate.weight', 'target_model.encoder.blocks.1.attention_layer.norm.weight', 'target_model.encoder.blocks.1.attention_layer.layer.q_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.k_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.v_proj.weight', 'target_model.encoder.blocks.1.attention_layer.layer.o_proj.weight', 'target_model.encoder.blocks.1.ff_layer.norm.weight', 'target_model.